# Hackathon IA PUCP 2026 · Starter notebook

Este notebook te lleva de cero a tu primera submission:

1. Cargar la data y entender cómo está organizada
2. Ver una hora de ejemplo (input vs. target vs. relieve)
3. Validar localmente con la misma métrica del leaderboard
4. **Baseline 1:** climatología por hora (score = 0)
5. **Baseline 2:** una CNN chica con el relieve (score ≈ 0.4)
6. Generar `submission.csv`

**Antes de correr en Kaggle:** *Settings → Accelerator → GPU T4*. En CPU también corre, pero más lento.

## 1. Cargar la data

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

**Cómo leer las dimensiones**

- `X`: `(horas, 17, 13, 4)` → pronóstico global GFS (~27 km). Fila 0 = norte, columna 0 = oeste.
- `Y`: `(horas, 70, 35, 4)` → modelo físico WRF (3 km). **Fila 0 = sur**, columna 0 = oeste.
- Las 4 variables, en este orden: humedad relativa (%), temperatura (K), viento U (este-oeste, m/s), viento V (norte-sur, m/s).
- `times`: hora en formato `YYYYMMDDHH`, en UTC (Perú = UTC−5). Hay algunos huecos sin datos.
- `static.npz`: el relieve (`dem`, metros) y las coordenadas de ambas grillas.

## 2. Una hora de ejemplo

In [ ]:
i = np.where(ttr == '2024050118')[0][0]   # 1 de mayo, 18 UTC (1 p. m. en Perú)
fig, ax = plt.subplots(1, 3, figsize=(13, 5))
ax[0].imshow(Xtr[i, ..., 1] - 273.15, cmap='coolwarm'); ax[0].set_title('Input GFS · T (°C)')
im = ax[1].imshow(np.flipud(Ytr[i, ..., 1]) - 273.15, cmap='coolwarm'); ax[1].set_title('Target WRF · T (°C)')
ax[2].imshow(np.flipud(dem), cmap='terrain'); ax[2].set_title('Relieve (m)')
plt.colorbar(im, ax=ax[1], shrink=0.7); plt.tight_layout(); plt.show()

El input no distingue valles de nevados; el target sí. Esa diferencia es lo que tu modelo tiene que aprender (y por eso el relieve ayuda mucho).

## 3. Validación local

El test es julio–agosto 2024. Para validar sin gastar submissions, separamos las **últimas dos semanas de junio** como validación.
Nunca uses un split aleatorio: horas consecutivas se parecen demasiado y te darían una validación optimista.

La métrica es la misma del leaderboard: 1 menos el promedio, en las 4 variables, del RMSE dividido entre el RMSE de la climatología. **Mayor es mejor; 0 = climatología, 1 = perfecto.**

In [ ]:
hour = lambda t: np.array([int(k[-2:]) for k in t])
val = ttr >= '2024061600'
trn = ~val

def climatology(Y, t):
    h = hour(t)
    return np.stack([Y[h == k].mean(0) for k in range(24)])   # (24, 70, 35, 4)

clim_trn = climatology(Ytr[trn], ttr[trn])
clim_rmse_val = np.sqrt(((clim_trn[hour(ttr[val])] - Ytr[val]) ** 2).mean(axis=(0, 1, 2)))

def local_score(pred, true):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=(0, 1, 2)))
    return 1 - (rmse / clim_rmse_val).mean(), dict(zip(VARS, rmse.round(3)))

print('Baseline 1 · climatología:', local_score(clim_trn[hour(ttr[val])], Ytr[val]))

## 4. Ventanas de 3 horas

El modelo recibe las horas t-2, t-1 y t del GFS para predecir el WRF en t. Si falta una hora previa (por un hueco en la data), repetimos la más cercana disponible.

In [ ]:
def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        prev = i
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (n, 3, 17, 13, 4)

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)
print(Wtr.shape, Wte.shape)

## 5. Baseline 2 · CNN chica con relieve

Idea simple:
1. Normalizamos el GFS y lo **interpolamos a la grilla de 3 km** usando las coordenadas reales (`grid_sample`).
2. Le sumamos el relieve y un mapa estático aprendido.
3. La red predice la **anomalía** respecto de la climatología de esa hora.

Hay mucho margen de mejora: arquitectura, más contexto temporal, pérdida por variable, ensambles, etc.

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
QUICK = os.environ.get('QUICK') == '1'     # solo para probar que todo corre
EPOCHS = 2 if QUICK else 25
torch.manual_seed(0); np.random.seed(0)

# Normalización (con estadísticas de train)
mu_x = Xtr[trn].mean((0, 1, 2)); sd_x = Xtr[trn].std((0, 1, 2))
clim_all = climatology(Ytr[trn], ttr[trn])
anom_sd = (Ytr[trn] - clim_all[hour(ttr[trn])]).std((0, 1, 2))

# Coordenadas de la grilla WRF dentro de la grilla GFS, para grid_sample
glat, glon = st['gfs_lats'], st['gfs_lons']            # GFS: fila 0 = norte
wlat, wlon = st['wrf_lats'], st['wrf_lons']            # WRF: fila 0 = sur
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=dev)   # (1,70,35,2)
DEM = torch.tensor((dem - dem.mean()) / dem.std(), dtype=torch.float32, device=dev)[None, None]

class Net(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(
            nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, w, 3, padding=1), nn.GELU(),
            nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, GRID.expand(B, -1, -1, -1), align_corners=True)   # (B,w,70,35)
        f = torch.cat([f, DEM.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

def to_t(a): return torch.tensor(a, dtype=torch.float32, device=dev)
CLIM = to_t(clim_all); ASD = to_t(anom_sd)

def prep(W, t):
    return to_t((W - mu_x) / sd_x), torch.tensor(hour(t), dtype=torch.long, device=dev)

def predict(model, W, t, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(W), bs):
            x, h = prep(W[s:s+bs], t[s:s+bs])
            out.append((CLIM[h] + model(x, h.float()) * ASD).cpu().numpy())
    return np.concatenate(out)

model = Net().to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * int(np.ceil(trn.sum() / 32)))
itr = np.where(trn)[0]
best, best_state = -9e9, None
for ep in range(EPOCHS):
    model.train(); np.random.shuffle(itr); t0 = time.time()
    for s in range(0, len(itr), 32):
        b = itr[s:s+32]
        x, h = prep(Wtr[b], ttr[b])
        y = (to_t(Ytr[b]) - CLIM[h]) / ASD
        loss = F.mse_loss(model(x, h.float()), y)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    sc, per_var = local_score(predict(model, Wtr[val], ttr[val]), Ytr[val])
    if sc > best: best, best_state = sc, {k: v.clone() for k, v in model.state_dict().items()}
    print(f'época {ep+1:02d} · loss {loss.item():.3f} · score val {sc:.4f} · {per_var} · {time.time()-t0:.0f}s')
model.load_state_dict(best_state)
print('Mejor score en validación:', round(best, 4))

## 6. Submission

Kaggle evalúa el test **cada 3 horas** (00, 03, …, 21 UTC). El archivo tiene una fila por píxel de cada hora evaluada:

`id = YYYYMMDDHH_fila_columna` (fila 0 = sur, columna 0 = oeste) y columnas `RH, T, U, V` en las mismas unidades del target.

In [ ]:
sample = pd.read_csv(find('sample_submission.csv'))
P = predict(model, Wte, tte)                                    # (horas_test, 70, 35, 4)
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS):
    sub[v] = P[ti, r, c, k].round(3)
sub.to_csv('submission.csv', index=False)
print(sub.shape); sub.head()

In [ ]:
# Guarda SOLO los pesos (sin el estado del optimizador) para usarlos en tu producto
os.makedirs('../models' if os.path.isdir('../models') else 'models', exist_ok=True)
ruta_pesos = os.path.join('../models' if os.path.isdir('../models') else 'models', 'modelo.pt')
torch.save(model.state_dict(), ruta_pesos)
print('Pesos guardados en', ruta_pesos, f'({os.path.getsize(ruta_pesos)/1e6:.1f} MB)')


## Ideas para mejorar

- **Viento (U, V)**: es donde más margen queda; el relieve manda. Prueba pérdidas por variable o más capacidad en el decoder.
- **Contexto temporal**: más horas de historia, o features de día del año.
- **Arquitecturas**: U-Net, ConvLSTM, atención.
- **Ensambles** de varias semillas.
- Y recuerda: el modelo es la mitad del reto. **Lleva tu modelo más allá de la métrica.**